# 05 · Statistics

The same procedure as v1 (`v1/stats/significance.py`):
- 95% percentile-bootstrap CIs over queries (10,000 resamples, seed 42)
- paired Wilcoxon signed-rank tests, Holm-corrected across all rows

For methods run with several shuffle seeds, each query's metric is first averaged over seeds, and the spread across seeds is reported separately.
- **R-precision is the primary metric.** MRR saturates on the random benchmark.
- **Outputs:** `v2/stats/results_<bench>.md` and figures in `v2/figures/`.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from src.figures import save, setup_style
setup_style()

In [ ]:
BENCHES = ["concept", "random"]
SPLIT = "test"
PAIRS = [
    ("llama", "qwen"), ("llama", "bge-reranker"), ("llama", "llama-pointwise"),
    ("llama-pointwise", "bge-reranker"), ("qwen→llama_k{k}_shuffled", "qwen"),
    ("llama", "qwen→llama_k{k}_shuffled"), ("bge-reranker", "qwen"), ("qwen", "minilm"),
    ("minilm", "tfidf"), ("specter2", "minilm"), ("qwen", "bm25"),
]

In [ ]:
import json
from src.pipeline_v2 import load_eval_sets, load_runs, select_sets
from src.stats_v2 import chance_levels, method_table, pairwise_tests, per_query, seed_spread

C = json.loads((RESEARCH / "v2" / "config.json").read_text())
TAG = f"{C['fields']}_{C['max_chars']}_{C['query_max_chars']}"
pairs = [(a.format(k=C["stage1_k"]), b.format(k=C["stage1_k"])) for a, b in PAIRS]
OUT = RESEARCH / "v2" / "stats"
OUT.mkdir(exist_ok=True)

In [ ]:
tables = {}
for bench in BENCHES:
    df = load_runs(bench, TAG, split=SPLIT)
    pq = per_query(df)
    chance = chance_levels(select_sets(load_eval_sets(bench), SPLIT))
    tables[bench] = {"methods": method_table(pq, chance), "spread": seed_spread(df),
                     "tests": pairwise_tests(pq, [p for p in pairs if {*p} <= set(pq["method"])])}
    print(f"=== {bench}  ({TAG}, {SPLIT})")
    display(tables[bench]["methods"])
    display(tables[bench]["spread"][tables[bench]["spread"]["seeds"] > 1])
    display(tables[bench]["tests"])

In [ ]:
def fmt(t):
    m = t["methods"]
    lines = ["| Method | R-precision % [95% CI] | MRR [95% CI] | nDCG@n | AP |", "|---|---|---|---|---|"]
    for _, r in m.iterrows():
        if r["method"].startswith("chance"):
            lines.append(f"| {r['method']} | {r['rprec']:.1f} | {r['mrr']:.3f} | | |")
        else:
            lines.append(f"| {r['method']} | {r['rprec']:.1f} [{r['rprec_lo']:.1f}, {r['rprec_hi']:.1f}] "
                         f"| {r['mrr']:.3f} [{r['mrr_lo']:.3f}, {r['mrr_hi']:.3f}] | {r['ndcg']:.3f} | {r['ap']:.3f} |")
    lines += ["", "| Comparison | Metric | Mean difference | p (Holm) |", "|---|---|---|---|"]
    for _, r in t["tests"].iterrows():
        lines.append(f"| {r['a']} vs {r['b']} | {r['metric']} | {r['mean_diff']:+.3f} | {r['p_holm']:.2g} |")
    return "\n".join(lines)

for bench, t in tables.items():
    md = f"# v2 statistics: {bench} benchmark ({SPLIT}, text config {TAG})\n\nGenerated by `v2/05_stats.ipynb`.\n\n" + fmt(t)
    (OUT / f"results_{bench}.md").write_text(md + "\n")
print("wrote", OUT / f"results_{bench}.md")

In [ ]:
from src.figures import fig_bench_compare, fig_forest
for bench, t in tables.items():
    save(fig_forest(t["methods"], bench), f"05_rprec_ci_{bench}")

**Figure (05_rprec_ci_<bench>).** Test-set R-precision per method, with 95% percentile-bootstrap CIs over queries; methods with several shuffle seeds are averaged per query first. Color marks the method family; the dashed line is the expected value of a random ranking.

In [ ]:
if {"random", "concept"} <= set(tables):
    save(fig_bench_compare(tables), "05_bench_compare")

**Figure (05_bench_compare).** The same methods on both benchmarks. The gap between the two points shows how much each method depends on negatives being topically unrelated.

### Findings

_Written from the outputs above after the notebook is run._